# Steel rust — baseline inference and validation
Mohammad Mango | M4U3

Loads the published **30-epoch YOLOv8n weights** and evaluates the same 88 validation images. **No training is performed.** Run all in a fresh Colab session. CPU is supported; GPU is optional. No API key, Drive mount or manual image upload is required.

Dataset and weights are downloaded from public GitHub Release URLs and checked against fixed SHA256 hashes. The exact deterministic 352/88 split is recreated using the training notebook's unchanged split code. The five personal photographs are downloaded from a pinned GitHub commit and SHA256-checked for a keyless demonstration and remain qualitative examples, not a labelled test set.

Expected outputs: metrics table, validation curves, ten validation predictions, five personal-photo predictions and a downloadable results ZIP. CPU evaluation may take several minutes; actual runtime is recorded. The previous embedded-photo version completed on CPU on 24 September 2026 in 52.43 seconds excluding installation. This revision only replaces embedded photos with verified public downloads of identical bytes; its complete Colab run has not yet been repeated.

The source dataset export declares CC BY 4.0. Sources: [University of Tebessa](https://universe.roboflow.com/university-of-tebessa/corrosion-eh3ms), [adapted version 1](https://universe.roboflow.com/mohammad-mango/structural-steel-rust-bboxes/dataset/1). Five personal photographs: Mohammad Mango. Current metrics do not support autonomous structural safety decisions.

Reference API documentation: [validation](https://docs.ultralytics.com/modes/val/), [prediction](https://docs.ultralytics.com/modes/predict/). This notebook pins the project's original Ultralytics version, rather than adopting newer documentation defaults.


## 1. Install the pinned package

In [ ]:
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'ultralytics==8.3.221'])

## 2. Configure CPU/GPU and output folders

In [ ]:
from pathlib import Path
import json, time, hashlib, zipfile, shutil, random, platform
from datetime import datetime, timezone
import torch, ultralytics, yaml, pandas as pd
from ultralytics import YOLO
from PIL import Image, ImageDraw
from IPython.display import display
from google.colab import files

FULL_TRAIN = False
MODEL = 'yolov8n.pt'
EPOCHS = 0
BATCH = 16
IMGSZ = 512
SEED = 42
CONF = 0.25
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
OUT = Path('/content/Steel_Rust_Inference') / RUN_ID
OUT.mkdir(parents=True, exist_ok=False)
WORK = Path('/content') / ('rust_inference_' + RUN_ID)
WORK.mkdir()
started = datetime.now(timezone.utc).isoformat()
t0 = time.perf_counter()
env = {'ultralytics':ultralytics.__version__, 'torch':torch.__version__, 'python':platform.python_version(),
       'hardware':torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU',
       'model':MODEL,'epochs':EPOCHS,'batch':BATCH,'imgsz':IMGSZ,'seed':SEED,'prediction_conf':CONF,
       'mode':'inference_only', 'weights_training_epochs':30, 'started_utc':started}
(OUT/'environment.json').write_text(json.dumps(env, indent=2))
(OUT/'pip_freeze.txt').write_text(subprocess.check_output([sys.executable,'-m','pip','freeze'],text=True))
print(env)
print('Output folder:', OUT)

## 3. Download and verify the frozen dataset

In [ ]:
import urllib.request
DATA_URL = "https://github.com/mnmango88/steel-rust-detection/releases/download/v1.0/steel-rust-v1-yolov8.zip.zip"
EXPECTED_SHA256 = "32220278b1b7277d1f662244014b876b7f8298925cec0569ded3decf08aff7b7"
archive = WORK/'source.zip'
try:
    urllib.request.urlretrieve(DATA_URL, archive)
    h = hashlib.sha256()
    with archive.open('rb') as f:
        for block in iter(lambda:f.read(1<<20), b''):
            h.update(block)
    source_sha = h.hexdigest()
    assert source_sha == EXPECTED_SHA256, f'Checksum mismatch: {source_sha}'
except Exception:
    archive.unlink(missing_ok=True)
    raise
(OUT/'source_sha256.txt').write_text(source_sha)
extract_root = WORK/'raw'
extract_root.mkdir(exist_ok=True)
with zipfile.ZipFile(archive) as z:
    for member in z.infolist():
        assert (extract_root/member.filename).resolve().is_relative_to(extract_root.resolve()), 'Unsafe ZIP path'
    z.extractall(extract_root)
candidates = [p for p in extract_root.rglob('data.yaml')
              if all((p.parent/s/'images').is_dir() for s in ['train','valid','test'])]
assert len(candidates)==1, f'Expected one dataset root, found {len(candidates)}'
raw = candidates[0].parent
print('Verified dataset:', source_sha)


## 4. Recreate the recorded 80/20 split

In [ ]:
extensions = {'.jpg','.jpeg','.png'}
original = {s:sorted(p for p in (raw/s/'images').iterdir() if p.suffix.lower() in extensions) for s in ['train','valid','test']}
assert [len(original[s]) for s in ['train','valid','test']] == [353,44,43], 'Unexpected source split; inspect before proceeding.'
assert yaml.safe_load((raw/'data.yaml').read_text())['names'] == ['rust']
move = random.Random(SEED).choice(original['train'])
assignments = [(p, s, 'train' if s=='train' and p!=move else 'valid') for s in original for p in original[s]]
prepared = WORK/'dataset_80_20'
for s in ['train','valid']:
    for kind in ['images','labels']:
        (prepared/s/kind).mkdir(parents=True)
records=[]; hashes={}; boxes=0
for p,old,new in assignments:
    label = p.parent.parent/'labels'/(p.stem+'.txt')
    assert label.exists(), f'Missing label: {p.name}'
    im=Image.open(p).convert('RGB'); im.load()
    pixel_hash=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
    assert pixel_hash not in hashes, f'Exact duplicate: {p.name} and {hashes.get(pixel_hash)}'
    hashes[pixel_hash]=p.name
    rows=[line for line in label.read_text().splitlines() if line.strip()]
    for row in rows:
        a=list(map(float,row.split()))
        assert len(a)==5 and a[0]==0 and all(0<=v<=1 for v in a[1:]) and min(a[3:])>0, (p.name,row)
    boxes+=len(rows)
    dest_name=old+'__'+p.name
    shutil.copy2(p,prepared/new/'images'/dest_name)
    shutil.copy2(label,prepared/new/'labels'/(Path(dest_name).stem+'.txt'))
    records.append({'original_split':old,'source_filename':p.name,'split':new,'filename':dest_name,'boxes':len(rows),'pixel_sha256':pixel_hash})
manifest=pd.DataFrame(records)
manifest.to_csv(OUT/'split_manifest.csv',index=False)
assert manifest['split'].value_counts().to_dict()=={'train':352,'valid':88}
assert boxes==4107
DATA=prepared/'data.yaml'
DATA.write_text(yaml.safe_dump({'path':str(prepared),'train':'train/images','val':'valid/images','names':{0:'rust'}}))
shutil.copy2(DATA,OUT/'data_run.yaml')
(OUT/'split_policy.txt').write_text('Original 353/44/43. Seed 42 selects one train image to move to validation; original valid and test merged into validation. Final 352/88. No independent test set. No label edits. Near-duplicate/site leakage not yet excluded. Moved image: '+move.name)
display(manifest.groupby('split').agg(images=('filename','count'),boxes=('boxes','sum')))
print('Label syntax and exact-duplicate checks passed. Moved image:',move.name)

## 5. Download and verify the trained weights

In [ ]:
WEIGHTS_URL = 'https://github.com/mnmango88/steel-rust-detection/releases/download/v1.0/best.pt'
WEIGHTS_SHA256 = '42c9db2e7e9f8e4a7b9d0702553d8f5728dac6d5cfd529a5893b0b40888857cb'
BEST = WORK / 'best.pt'
urllib.request.urlretrieve(WEIGHTS_URL, BEST)
h = hashlib.sha256()
with BEST.open('rb') as f:
    for block in iter(lambda: f.read(1 << 20), b''):
        h.update(block)
if h.hexdigest() != WEIGHTS_SHA256:
    BEST.unlink(missing_ok=True)
    raise ValueError('Weights checksum mismatch; download stopped.')
print('Verified trained weights. No training will run.')


## 6. Evaluate the 88 validation images
Baseline reference: P 0.355081; R 0.364922; mAP50 0.292277; mAP50–95 0.122363. Small numerical differences across hardware/software environments are possible. These are validation metrics, not independent test metrics.

In [ ]:
best=YOLO(str(BEST))
metrics=best.val(data=str(DATA),split='val',imgsz=IMGSZ,batch=BATCH,device=DEVICE,plots=True,
                 project=str(OUT),name='validation',exist_ok=False)
table=pd.DataFrame([{'precision':float(metrics.box.mp),'recall':float(metrics.box.mr),
                     'mAP50':float(metrics.box.map50),'mAP50_95':float(metrics.box.map)}])
table.to_csv(OUT/'metrics.csv',index=False)
display(table)
print('Metrics are in the range 0–1.')

## 7. Save ten validation predictions

In [ ]:
samples=sorted((prepared/'valid'/'images').glob('*'))[:10]
best.predict(source=[str(p) for p in samples],imgsz=IMGSZ,conf=CONF,device=DEVICE,
             save=True,save_txt=True,save_conf=True,project=str(OUT/'evidence'),name='validation_predictions')
(OUT/'validation_sample_names.txt').write_text('\n'.join(p.name for p in samples))
for p in sorted((OUT/'evidence'/'validation_predictions').glob('*.jpg')):
    display(Image.open(p).resize((384,384)))

## 8. Run inference on five author photographs
Confidence 0.25. Preserve aspect ratio when displaying images. No new-image accuracy metric is claimed.

In [ ]:
from urllib.parse import quote
PHOTO_BASE = 'https://raw.githubusercontent.com/mnmango88/steel-rust-detection/e24d29c9d53466b23c70f7f36d7ef726218a80ee/results/evidence/new_image_inputs/'
PHOTO_SHA256 = {'Image (39).jpg': 'ca95f58d059276db2877dd5837e231532d07f7e24a21f77506904a20ecd2831d', 'Image (41).jpg': '2c76f3bffc9c44a6e1bb939c0713dd8fd6fbfb9098c338b48e7b4c295da0494c', 'Image (42).jpg': 'ab7d9a9f2193e6e3c75fde59bc581ed6814b3a220a3e351080ccc6203e6388b6', 'Image (45).jpg': '6a1c5ad348e3371f5acd7677f6303fde925497c2854b60badafc248b9c88be49', 'Image (49).jpg': '097f6c72905ab3bff66890fe09e8509bf73ac9aa51e8e0a271f48d77967ced52'}

new_dir=WORK/'new_images'
new_dir.mkdir(exist_ok=True)
accepted=[]
for name, expected_sha in PHOTO_SHA256.items():
    p=new_dir/name
    urllib.request.urlretrieve(PHOTO_BASE + quote(name), p)
    assert hashlib.sha256(p.read_bytes()).hexdigest() == expected_sha, f'Photo checksum mismatch: {name}'
    im=Image.open(p).convert('RGB')
    ph=hashlib.sha256(str(im.size).encode()+im.tobytes()).hexdigest()
    assert ph not in hashes, f'{name} is already in the dataset'
    accepted.append(str(p))
best.predict(source=accepted,imgsz=IMGSZ,conf=CONF,device=DEVICE,save=True,save_txt=True,save_conf=True,
             project=str(OUT/'evidence'),name='new_image_predictions')
shutil.copytree(new_dir,OUT/'evidence'/'new_image_inputs',dirs_exist_ok=True)
for p in sorted((OUT/'evidence'/'new_image_predictions').glob('*.jpg')):
    display(Image.open(p))


## 9. Save the run record and download outputs

In [ ]:
proof = {**env, 'finished_utc': datetime.now(timezone.utc).isoformat(),
         'session_seconds': time.perf_counter()-t0, 'completed_training_epochs_this_run': 0,
         'weights_training_epochs': 30, 'weights_sha256': WEIGHTS_SHA256,
         'source_sha256': source_sha, 'split': {'train':352, 'valid':88}}
(OUT/'run_record.json').write_text(json.dumps(proof, indent=2))
zip_path=shutil.make_archive(str(OUT.parent/('Steel_Rust_Inference_'+RUN_ID)), 'zip', OUT)
print('Outputs:', OUT)
print('Download:', zip_path)
files.download(zip_path)
